In [ ]:
import json

train = json.load(open("taskA/train.json", "r", encoding="utf-8"))
test = json.load(open("taskA/test.json", "r", encoding="utf-8"))

print("train:", len(train))
print("test:", len(test))
print("train ids appearing in test:", len(set(train.keys()).intersection(set(test.keys()))))

print("\n")

print("Avg text length in train:", sum(len(content["text"]) for id, content in train.items()) / len(train))
print("Avg text length in test:", sum(len(content["text"]) for id, content in test.items()) / len(test))

In [ ]:
from statistics import median, quantiles


def print_words_stats(data):
    word_counts_per_doc = []

    for document in data.values():
        text = document["text"]

        if isinstance(text, str):
            word_count = len(text.split())
        else:
            word_count = sum(len(sentence.split()) for sentence in text)

        word_counts_per_doc.append(word_count)

    print("\n=== Dataset Statistics ===")

    n = len(word_counts_per_doc)
    print(f"Total documents: {n}")

    if n == 0:
        print("No documents to analyze.")
        return

    total_words = sum(word_counts_per_doc)

    if n == 1:
        q1 = q3 = word_counts_per_doc[0]
    else:
        q1, _, q3 = quantiles(
            word_counts_per_doc,
            n=4,
            method="inclusive",
        )

    print(f"Average words per document: {total_words / n:.2f}")
    print(f"Max words per document: {max(word_counts_per_doc)}")
    print(f"Min words per document: {min(word_counts_per_doc)}")
    print(f"Total words in dataset: {total_words}")
    print(f"Median words per document: {median(word_counts_per_doc):g}")
    print(f"25th percentile (Q1): {q1:g}")
    print(f"75th percentile (Q3): {q3:g}")
    print(f"Interquartile range (IQR): {q3 - q1:g}")

    bins = [
        ("0-50", 50),
        ("51-100", 100),
        ("101-200", 200),
        ("201-500", 500),
        ("501-1000", 1000),
        ("1001-1499", 1499),
        #("1500-1999", 1999),
        #("2000-2500", 2500),
        #("2501-2999", 2999),
        #("3000+", float("inf")),
    ]
    bin_counts = [0] * len(bins)

    for word_count in word_counts_per_doc:
        for i, (_, upper_bound) in enumerate(bins):
            if word_count <= upper_bound:
                bin_counts[i] += 1
                break

    print("\n=== Word Count Distribution ===")
    for (label, _), count in zip(bins, bin_counts):
        percentage = count / n * 100
        print(f"{label} words: {count} documents ({percentage:.1f}%)")

In [ ]:
print("\nTrain dataset word count statistics:")
print_words_stats(train)

In [ ]:
print("\nTest dataset word count statistics:")
print_words_stats(test)

In [ ]:
def check_terms(data):
    for pmid, content in data.items():
        text = content["text"]
        for term in content["terms"]:
            if text.find(term) == -1:
                print(f"Term '{term}' not found in document {pmid}")

check_terms(train)
check_terms(test)

In [ ]:
def print_terms_stats(data):
    term_counts_per_doc = []
    unique_terms = set()

    for document in data.values():
        terms = document["terms"]
        term_count = len(terms)
        term_counts_per_doc.append(term_count)
        unique_terms.update(term.lower() for term in terms)

    print("\n=== Term Statistics ===")

    n = len(term_counts_per_doc)
    print(f"Total documents: {n}")

    if n == 0:
        print("No documents to analyze.")
        return

    total_terms = sum(term_counts_per_doc)

    if n == 1:
        q1 = q3 = term_counts_per_doc[0]
    else:
        q1, _, q3 = quantiles(
            term_counts_per_doc,
            n=4,
            method="inclusive",
        )

    print(f"Average terms per document: {total_terms / n:.2f}")
    print(f"Max terms per document: {max(term_counts_per_doc)}")
    print(f"Min terms per document: {min(term_counts_per_doc)}")
    print(f"Total terms in dataset: {total_terms}")
    print(f"Unique terms in dataset: {len(unique_terms)}")
    print(f"Median terms per document: {median(term_counts_per_doc):g}")
    print(f"25th percentile (Q1): {q1:g}")
    print(f"75th percentile (Q3): {q3:g}")
    print(f"Interquartile range (IQR): {q3 - q1:g}")

In [ ]:
print("\nTrain dataset term statistics:")
print_terms_stats(train)

In [ ]:
print("\nTest dataset term statistics:")
print_terms_stats(test)